In [0]:
# Teste de ambiente PySpark no Databricks
from pyspark.sql import functions as F

data = [("Porto do Rio de Janeiro", 12.5), ("Porto de Santos", 14.2), ("Porto de Paranaguá", 13.0)]
columns = ["nome_porto", "calado_maximo_m"]

df_teste = spark.createDataFrame(data, schema=columns)
display(df_teste.withColumn("status_operacional", F.when(F.col("calado_maximo_m") >= 13.0, "Profundo").otherwise("Restrito")))

In [0]:
# ==============================================================================
# PASSO 1 (CORRIGIDO): INGESTÃO E CRIAÇÃO DA CAMADA BRONZE (RAW DATA)
# ==============================================================================

import pandas as pd
from pyspark.sql import functions as F

# 1. Criar o Banco de Dados para a Camada Bronze no Catálogo
spark.sql("CREATE DATABASE IF NOT EXISTS mvp_s3_bronze")

# 2. URL do arquivo Raw no seu GitHub
url_dados_antaq = "https://raw.githubusercontent.com/ThiagoFOPinto/puc-data-science-analytics/main/mvp_s2_predictive/data/puc_infra_portos.csv"

try:
    print("Baixando e lendo o arquivo do GitHub via Pandas...")
    
    # Lemos a URL HTTP diretamente para a memória via Pandas
    df_pd = pd.read_csv(url_dados_antaq)
    
    # Padronizar nomes das colunas (em minúsculas e sem espaços)
    df_pd.columns = [c.strip().lower().replace(" ", "_") for c in df_pd.columns]
    
    # Converter para DataFrame do PySpark
    df_bronze_raw = spark.createDataFrame(df_pd)
    
    print("✅ Base do GitHub baixada e convertida para PySpark com sucesso!")

except Exception as e:
    print(f"Erro ao carregar dados: {e}")

# 3. Adicionar Metadados de Ingestão e Governança
df_bronze = df_bronze_raw \
    .withColumn("dh_ingestao", F.current_timestamp()) \
    .withColumn("fonte_dados", F.lit("GitHub_puc_infra_portos"))

# 4. Salvar na Camada Bronze (Tabela Delta)
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_s3_bronze.infra_portos")

print("✅ Ingestão da Camada Bronze concluída com sucesso!")
print(f"Total de registros salvos na tabela Bronze: {spark.table('mvp_s3_bronze.infra_portos').count()}")

# Exibir prévia dos dados salvos no catálogo
display(spark.table("mvp_s3_bronze.infra_portos").limit(5))

In [0]:
# ==============================================================================
# PASSO 2: CAMADA SILVER - LIMPEZA, TIPAGEM E QUALIDADE DE DADOS
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

# 1. Criar o Banco de Dados para a Camada Silver
spark.sql("CREATE DATABASE IF NOT EXISTS mvp_s3_silver")

# 2. Ler os dados brutos da Camada Bronze
df_bronze_input = spark.table("mvp_s3_bronze.infra_portos")

print(f"📊 Registros na Camada Bronze (Entrada): {df_bronze_input.count()}")

# 3. Tratamentos de Qualidade e Padronização na Silver
df_silver_clean = (
    df_bronze_input
    # A. Unicidade: Remoção de registros duplicados
    .dropDuplicates()
    
    # B. Padronização de Texto (Porto com inicial maiúscula e sem espaços extras)
    .withColumn("porto", F.initcap(F.trim(F.col("porto"))))
    
    # C. Tipagem Adequada (Conversão de métricas numéricas para Double/Integer)
    .withColumn("dwt_medio", F.col("dwt_medio").cast(DoubleType()))
    .withColumn("ano", F.col("ano").cast(IntegerType()))
    .withColumn("mes", F.col("mes").cast(IntegerType()))
    
    # D. Validação de Acurácia: Garantir DWT válido (positivo ou nulo)
    .filter((F.col("dwt_medio").isNull()) | (F.col("dwt_medio") >= 0))
    
    # E. Metadados da Silver
    .withColumn("dh_processamento_silver", F.current_timestamp())
)

# 4. Salvar persistentemente na Camada Silver (Tabela Delta)
df_silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_s3_silver.infra_portos")

print("✅ Camada Silver processada e salva com sucesso!")
print(f"📊 Registros validados na Camada Silver: {spark.table('mvp_s3_silver.infra_portos').count()}")

# Exibir prévia dos dados limpos
display(spark.table("mvp_s3_silver.infra_portos").limit(5))